# Fragmented Identity Resolution Pipeline

End-to-end pipeline for entity resolution on NC Voter Registration data.

## Pipeline stages:
1. **Load & Preprocess** — Normalize names, addresses, ZIP codes
2. **Silver/Gold Label Construction** — Build proxy ground truth from exact-match groups
3. **Pair Generation** — Positive, hard-negative, and block-negative pairs with entity-level splitting
4. **Synthetic Augmentation** — Rule-based perturbations (replacing LLM approach)
5. **Model Training** — Siamese char-CNN for pairwise match scoring
6. **Evaluation** — Precision, Recall, F1, AUC, calibration analysis

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")

# Ensure src is importable
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import normalize_dataframe
from src.label_construction import (
    build_silver_groups,
    validate_gold_groups,
    load_existing_pairs,
)
from src.pair_generation import (
    generate_gold_positive_pairs,
    generate_hard_negative_pairs,
    generate_block_negatives,
    generate_address_change_pairs,
    generate_synthetic_hard_negatives,
    pairs_from_dpl,
    pairs_from_ndpl,
    combine_pairs,
    entity_level_split,
)
from src.augmentation import augment_dataframe, generate_augmented_pairs, PerturbationType
from src.dataset import RecordPairDataset
from src.model import SiameseMatchingNetwork
from src.train import train_model, evaluate
from src.inference import DuplicateDetector

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 1. Load and Preprocess Data

We load the raw NCVoters TSV (Alamance County, ~14K records) and normalize:
- **Names**: uppercase, strip punctuation, collapse whitespace
- **Addresses**: normalize street suffixes (STREET→ST), directionals (NORTH→N), unit markers
- **ZIP**: extract ZIP5
- **Birth year**: derived from `age` column (snapshot year 2018)

In [ ]:
# Load raw data
raw_df = pd.read_csv(
    "ncvoters.tsv",
    sep="\t",
    dtype=str,
    na_filter=False,
    encoding="utf-8",
)
raw_df = raw_df.loc[:, ~raw_df.columns.str.contains(r"^Unnamed")]
print(f"Raw records: {len(raw_df):,}")
print(f"Columns: {list(raw_df.columns[:10])}... ({len(raw_df.columns)} total)")

In [ ]:
# Normalize
df = normalize_dataframe(raw_df, snapshot_year=2018)
print(f"Normalized records: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
# Quick data quality check
print("=== Data Quality ===")
for col in df.columns:
    non_empty = (df[col].astype(str).str.strip() != "").sum()
    print(f"  {col:20s}: {non_empty:>6,} / {len(df):,} ({100*non_empty/len(df):.1f}%)")

## 2. Construct Silver & Gold Labels

**Silver groups**: exact match on `(first, middle, last, suffix, street, city, zip5)` → records with identical normalized identity fields but distinct internal IDs.

**Gold groups**: silver groups validated by `birth_year` — all members share the same birth year = high-confidence same person.

**Birth-year conflicts**: silver groups where birth years disagree = **hard negatives** (likely different people at same address, e.g., parent/child).

In [ ]:
# Build silver groups
df = build_silver_groups(df)

n_silver = (df["silver_group_id"] >= 0).sum()
n_groups = df[df["silver_group_id"] >= 0]["silver_group_id"].nunique()
print(f"Records in silver duplicate groups: {n_silver:,}")
print(f"Number of silver groups: {n_groups:,}")

In [ ]:
# Validate gold groups
gold_df, conflict_df, stats = validate_gold_groups(df)

print("=== Label Construction Stats ===")
for k, v in stats.items():
    print(f"  {k}: {v}")

print(f"\nGold positive records: {len(gold_df):,}")
print(f"Hard negative records (birth-year conflict): {len(conflict_df):,}")

In [ ]:
# Inspect a few gold groups
print("=== Sample Gold Group (same person, high confidence) ===")
sample_gold_gid = gold_df["silver_group_id"].unique()[0]
display(gold_df[gold_df["silver_group_id"] == sample_gold_gid][
    ["id", "first_name", "middle_name", "last_name", "street_address", "city", "zip5", "birth_year"]
])

if len(conflict_df) > 0:
    print("\n=== Sample Conflict Group (same name+address, different birth year → hard negative) ===")
    sample_conflict_gid = conflict_df["silver_group_id"].unique()[0]
    display(conflict_df[conflict_df["silver_group_id"] == sample_conflict_gid][
        ["id", "first_name", "middle_name", "last_name", "street_address", "city", "zip5", "birth_year"]
    ])

## 3. Generate Training & Evaluation Pairs

**Sources:**
- Gold groups → positive pairs (label=1)
- Birth-year conflict groups → hard negative pairs (label=0)
- Block negatives (same ZIP + last-name prefix, different person) → label=0
- **Synthetic hard negatives** (chimeric records mixing attributes between people) → label=0
- Existing DPL/NDPL ground truth (from HPI dataset)

**Synthetic hard negatives** are generated by:
1. Same-last-name swap: person A's name + person B's address → confusing near-miss
2. First-name collision: different people with same first name, different last name
3. Attribute-mix: random cross-pollination of name/address fields between strangers

This forces the model to verify ALL fields holistically, not rely on any single field.

**Critical**: We split at the **entity/group level**, not pair level, to prevent leakage.

In [ ]:
# Generate pairs from our silver/gold construction
gold_pos_pairs = generate_gold_positive_pairs(gold_df)
hard_neg_pairs = generate_hard_negative_pairs(conflict_df)
block_neg_pairs = generate_block_negatives(df, n_negatives=5000)
addr_change_pairs, addr_synthetic = generate_address_change_pairs(gold_df, df, n_pairs=3000)

# NEW: Generate synthetic hard negatives (confusing near-misses)
syn_hard_neg_pairs, syn_hard_neg_records = generate_synthetic_hard_negatives(
    df, n_negatives=5000, random_state=42
)

print(f"Gold positive pairs: {len(gold_pos_pairs):,}")
print(f"Hard negative pairs: {len(hard_neg_pairs):,}")
print(f"Block negative pairs: {len(block_neg_pairs):,}")
print(f"Address-change pairs (moved person): {len(addr_change_pairs):,}")
print(f"Synthetic hard negative pairs: {len(syn_hard_neg_pairs):,}")
print(f"  → Chimeric synthetic records created: {len(syn_hard_neg_records):,}")

In [ ]:
# Load existing DPL/NDPL ground truth
dpl, ndpl = load_existing_pairs("ncvoters_DPL.tsv", "ncvoters_NDPL.tsv")
dpl_pairs = pairs_from_dpl(dpl)
ndpl_pairs = pairs_from_ndpl(ndpl, max_pairs=10000)

print(f"DPL positive pairs: {len(dpl_pairs):,}")
print(f"NDPL negative pairs (sampled): {len(ndpl_pairs):,}")

In [ ]:
# Combine all pair sources (now including synthetic hard negatives)
all_pairs = combine_pairs(
    gold_pos_pairs, hard_neg_pairs, block_neg_pairs,
    addr_change_pairs, syn_hard_neg_pairs, dpl_pairs, ndpl_pairs,
)

print(f"\nTotal combined pairs: {len(all_pairs):,}")
print(f"  Positives (label=1): {(all_pairs['label'] == 1).sum():,}")
print(f"  Negatives (label=0): {(all_pairs['label'] == 0).sum():,}")
print(f"\nPair sources:")
print(all_pairs.groupby(["source", "label"]).size().unstack(fill_value=0))

In [ ]:
# Entity-level train/test split (NO leakage)
train_pairs, test_pairs = entity_level_split(all_pairs, df, test_size=0.2)

print(f"Train pairs: {len(train_pairs):,} (pos={int((train_pairs['label']==1).sum())}, neg={int((train_pairs['label']==0).sum())})")
print(f"Test  pairs: {len(test_pairs):,} (pos={int((test_pairs['label']==1).sum())}, neg={int((test_pairs['label']==0).sum())})")

## 4. Synthetic Augmentation (Rule-Based)

Instead of using an LLM (slow, unreliable), we apply **controlled, reproducible perturbations**:

| Category | Perturbations | Example |
|----------|--------------|--------|
| **Name** | Nicknames, initials, typos, transpositions, hyphen changes, **phonetic subs**, **truncation** | WILLIAM → BILL, ROBERT → ROEBRT, JOHNSON → JONSON, CHRISTOPHER → CHRISTO |
| **Name (test-only)** | **Married names**, **case noise** | SMITH → SMITH-JONES, ROBERT → robert |
| **Address** | Suffix swaps, unit format changes, directional expansion | STREET → ST, APT 2B → #2B |
| **Missingness** | Blank middle name, suffix, unit | ANNA → (empty) |

**Anti-overfitting**: Some perturbation types are **held out from training** and only used in test evaluation. This proves the model generalizes beyond the specific augmentation patterns.

In [ ]:
# Generate training augmentations (uses TRAIN_TYPES only)
train_seed, train_synthetic = augment_dataframe(
    df, mode="train", frac=0.15,
    n_variants_per_record=2,
    n_perturbations_per_variant=2,
    random_state=42,
)

print(f"Training augmentation:")
print(f"  Seed records: {len(train_seed):,}")
print(f"  Synthetic records: {len(train_synthetic):,}")

# Show perturbation type distribution
all_ptypes = train_synthetic["perturbation_types"].str.split(",").explode()
print(f"\nPerturbation type distribution (training):")
print(all_ptypes.value_counts().to_string())

In [ ]:
# Show some augmentation examples
print("=== Augmentation Examples ===")
fields = ["first_name", "middle_name", "last_name", "street_address", "city", "zip5"]

for seed_id in train_seed["id"].head(5):
    seed_row = train_seed[train_seed["id"] == seed_id].iloc[0]
    syn_rows = train_synthetic[train_synthetic["seed_id"] == seed_id]
    
    print(f"\n--- Seed: {seed_id} ---")
    print(f"  Original: {' | '.join(str(seed_row[f]) for f in fields)}")
    for _, syn_row in syn_rows.iterrows():
        ptypes = syn_row['perturbation_types']
        print(f"  Variant:  {' | '.join(str(syn_row[f]) for f in fields)}  [{ptypes}]")

In [ ]:
# Generate test augmentations (includes HELD-OUT types for stress testing)
test_seed, test_synthetic = augment_dataframe(
    df, mode="test", frac=0.05,
    n_variants_per_record=2,
    n_perturbations_per_variant=2,
    random_state=99,
)

# Also generate stress-test set with ONLY held-out perturbation types
stress_seed, stress_synthetic = augment_dataframe(
    df, mode="test_only", frac=0.03,
    n_variants_per_record=2,
    n_perturbations_per_variant=2,
    random_state=123,
)

print(f"Test augmentation: {len(test_synthetic):,} synthetic records")
print(f"Stress-test augmentation (held-out types only): {len(stress_synthetic):,} synthetic records")

In [ ]:
# Add synthetic augmented records to the record pool and generate their pairs
# Give synthetic records unique IDs
train_synthetic["id"] = [f"syn_train_{i}" for i in range(len(train_synthetic))]
test_synthetic["id"] = [f"syn_test_{i}" for i in range(len(test_synthetic))]
stress_synthetic["id"] = [f"syn_stress_{i}" for i in range(len(stress_synthetic))]

# Generate augmented pairs (each synthetic ↔ its seed = positive)
aug_train_pairs = generate_augmented_pairs(train_seed, train_synthetic)
aug_test_pairs = generate_augmented_pairs(test_seed, test_synthetic)
aug_stress_pairs = generate_augmented_pairs(stress_seed, stress_synthetic)

print(f"Augmented train pairs: {len(aug_train_pairs):,}")
print(f"Augmented test pairs: {len(aug_test_pairs):,}")
print(f"Augmented stress pairs: {len(aug_stress_pairs):,}")

In [ ]:
# Build expanded record pool (original + all synthetic)
all_records = pd.concat([
    df,
    addr_synthetic.reindex(columns=df.columns),
    syn_hard_neg_records.reindex(columns=df.columns),  # chimeric hard negatives
    train_synthetic.reindex(columns=df.columns),
    test_synthetic.reindex(columns=df.columns),
    stress_synthetic.reindex(columns=df.columns),
], ignore_index=True)

# Final train pairs = original train + augmented train
final_train_pairs = pd.concat([train_pairs, aug_train_pairs], ignore_index=True)
# Final test pairs = original test + augmented test
final_test_pairs = pd.concat([test_pairs, aug_test_pairs], ignore_index=True)

print(f"All records (original + synthetic): {len(all_records):,}")
print(f"Final train pairs: {len(final_train_pairs):,}")
print(f"Final test pairs: {len(final_test_pairs):,}")
print(f"Stress test pairs: {len(aug_stress_pairs):,}")

## 5. Build Datasets & Train Siamese Network

**Architecture**: Character-level CNN encoder (shared weights) → pairwise comparison → MLP classifier

This addresses the embedding concern directly: character-level representations naturally handle typos, abbreviations, and nickname variations — which pretrained LMs like BERT aren't designed for.

In [ ]:
# Build PyTorch datasets
train_dataset = RecordPairDataset(final_train_pairs, all_records, max_len=128)
test_dataset = RecordPairDataset(final_test_pairs, all_records, max_len=128, char_vocab=train_dataset.char_vocab)

print(f"Character vocabulary size: {train_dataset.vocab_size}")
print(f"Train samples: {len(train_dataset):,}")
print(f"Test  samples: {len(test_dataset):,}")

# Show a sample
sample = train_dataset[0]
print(f"\nSample input1 shape: {sample['input1'].shape}")
print(f"Sample input2 shape: {sample['input2'].shape}")
print(f"Sample label: {sample['label'].item()}")

In [ ]:
# Create data loaders
BATCH_SIZE = 128

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)}")
print(f"Test  batches: {len(test_loader)}")

In [ ]:
# Initialize model
model = SiameseMatchingNetwork(
    vocab_size=train_dataset.vocab_size,
    embed_dim=32,
    num_filters=128,
    kernel_sizes=(3, 4, 5),
    encoder_output_dim=128,
    classifier_hidden_dim=64,
    dropout=0.3,
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")
print(model)

In [ ]:
# Train (early stopping on F1 — better for imbalanced data)
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    device=DEVICE,
    n_epochs=30,
    lr=1e-3,
    weight_decay=1e-4,
    patience=7,
    best_metric="f1",
)

## 6. Evaluation

We evaluate on:
1. **Standard test set** — original pairs + augmented pairs (train-type perturbations)
2. **Stress test** — only held-out perturbation types (model has never seen these patterns)

**Metrics**: Precision, Recall, F1, AUC, FPR, FNR across multiple thresholds.

In [ ]:
import torch.nn as nn

criterion = nn.BCEWithLogitsLoss()

# Standard test evaluation
print("=== Standard Test Evaluation ===")
test_metrics = evaluate(model, test_loader, criterion, DEVICE)
for k, v in test_metrics.items():
    print(f"  {k:12s}: {v:.4f}")

In [ ]:
# Stress test evaluation (held-out perturbation types only)
if len(aug_stress_pairs) > 0:
    stress_dataset = RecordPairDataset(
        aug_stress_pairs, all_records, max_len=128,
        char_vocab=train_dataset.char_vocab
    )
    stress_loader = DataLoader(stress_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    print("=== Stress Test Evaluation (held-out perturbation types) ===")
    stress_metrics = evaluate(model, stress_loader, criterion, DEVICE)
    for k, v in stress_metrics.items():
        print(f"  {k:12s}: {v:.4f}")
    
    print("\nThis tests generalization beyond the specific augmentation patterns used in training.")
else:
    print("No stress test pairs generated.")

In [ ]:
# Training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].set_xlabel("Epoch")

axes[1].plot(history["val_f1"])
axes[1].set_title("Validation F1")
axes[1].set_xlabel("Epoch")

axes[2].plot(history["val_auc"])
axes[2].set_title("Validation AUC")
axes[2].set_xlabel("Epoch")

axes[3].plot(history["val_precision"], label="Precision")
axes[3].plot(history["val_recall"], label="Recall")
axes[3].set_title("Precision / Recall")
axes[3].legend()
axes[3].set_xlabel("Epoch")

plt.tight_layout()
plt.show()

In [ ]:
# Threshold analysis — operational risk trade-off
print("=== Threshold Analysis ===")
print(f"{'Threshold':>10s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s} {'FPR':>10s} {'FNR':>10s}")
print("-" * 60)

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    metrics = evaluate(model, test_loader, criterion, DEVICE, threshold=threshold)
    print(
        f"{threshold:>10.1f} "
        f"{metrics['precision']:>10.4f} "
        f"{metrics['recall']:>10.4f} "
        f"{metrics['f1']:>10.4f} "
        f"{metrics['fpr']:>10.4f} "
        f"{metrics['fnr']:>10.4f}"
    )

In [ ]:
# Save model with full metadata for inference
torch.save({
    "model_state_dict": model.state_dict(),
    "char_vocab": train_dataset.char_vocab,
    "history": history,
    "model_config": {
        "vocab_size": train_dataset.vocab_size,
        "embed_dim": 32,
        "num_filters": 128,
        "kernel_sizes": (3, 4, 5),
        "encoder_output_dim": 128,
        "classifier_hidden_dim": 64,
        "dropout": 0.3,
    },
    "fields": train_dataset.fields,
    "max_len": train_dataset.max_len,
}, "siamese_model.pt")

print("Model saved to siamese_model.pt")

## Summary

This pipeline implements:

1. **Preprocessing**: Name/address/ZIP normalization with suffix, directional, and unit canonicalization
2. **Silver/Gold labels**: Proxy ground truth from exact-match groups + birth-year validation
3. **Pair generation**: Positives, hard negatives, block negatives, **synthetic hard negatives (chimeric records)**, and DPL/NDPL integration with entity-level splitting
4. **Synthetic augmentation**: Rule-based perturbations (nicknames, typos, phonetic subs, married names, truncation, case noise) with train/test-held-out split
5. **Siamese char-CNN**: Character-level embeddings → CNN encoder → pairwise comparison → match probability
6. **Two-stage evaluation**: Candidate retrieval (Recall@K) + pair classification (F1, AUC, FPR/FNR)
7. **Inference pipeline**: `DuplicateDetector` class for production-ready duplicate detection on unseen data

### Key design decisions:
- **Rule-based augmentation** instead of LLM-based: faster, reproducible, controllable
- **Character-level embeddings**: naturally handles typos, abbreviations, nicknames — better than pretrained LMs for this task
- **Synthetic hard negatives**: chimeric records (name A + address B) force the model to verify ALL fields
- **Anti-overfitting**: held-out perturbation types prove generalization
- **Entity-level splitting**: prevents leakage between train/test
- **F1-based early stopping**: more relevant than loss for imbalanced data (9:1 negative:positive ratio)
- **Production inference**: `DuplicateDetector.from_checkpoint()` → normalize → embed → retrieve → score

In [ ]:
@torch.no_grad()
def embed_records(model, dataset, all_ids, device, batch_size=256):
    """Embed all records using the trained encoder."""
    model.eval()
    embeddings = []
    
    # Encode each record individually
    for rid in all_ids:
        text = dataset._record_to_text(str(rid))
        enc = dataset._encode(text).unsqueeze(0).to(device)
        emb = model.encode(enc)
        embeddings.append(emb.cpu())
    
    return torch.cat(embeddings, dim=0)

# Get embeddings for all original records
original_ids = df["id"].tolist()
record_embeddings = embed_records(model, train_dataset, original_ids, DEVICE)
print(f"Embedded {len(record_embeddings)} records, dim={record_embeddings.shape[1]}")

In [ ]:
def recall_at_k(embeddings, ids, dpl_pairs, k_values=[1, 5, 10, 20, 50]):
    """Compute Recall@K for candidate retrieval."""
    # Build ground truth lookup
    gt = {}
    for _, row in dpl_pairs.iterrows():
        id1, id2 = str(row["id1"]), str(row["id2"])
        gt.setdefault(id1, set()).add(id2)
        gt.setdefault(id2, set()).add(id1)

    id_to_idx = {str(rid): i for i, rid in enumerate(ids)}
    
    # Compute cosine similarity matrix
    normed = embeddings / embeddings.norm(dim=1, keepdim=True).clamp(min=1e-8)
    sim_matrix = normed @ normed.T
    
    results = {k: [] for k in k_values}
    
    for query_id, true_matches in gt.items():
        if query_id not in id_to_idx:
            continue
        qi = id_to_idx[query_id]
        sims = sim_matrix[qi].clone()
        sims[qi] = -1  # exclude self
        
        top_indices = sims.argsort(descending=True)
        top_ids = [ids[idx] for idx in top_indices[:max(k_values)]]
        
        for k in k_values:
            retrieved = set(str(rid) for rid in top_ids[:k])
            hit = len(retrieved & true_matches) > 0
            results[k].append(hit)
    
    print(f"\n=== Recall@K (n_queries={len(list(gt.keys()))}) ===")
    for k in k_values:
        r = np.mean(results[k]) if results[k] else 0
        print(f"  Recall@{k:<3d}: {r:.4f}")

recall_at_k(record_embeddings, original_ids, dpl)

## 8. Inference on Unseen Data — Duplicate Detection Demo

This demonstrates using the trained model to detect duplicates in **new, unaugmented** records that the model has never seen. This is the production use case:

1. A batch of new voter registrations arrives
2. The system normalizes them
3. Finds candidate matches against the existing database
4. Scores each pair with the Siamese model
5. Returns ranked matches with confidence scores

This also simulates the real-world scenario described in the project spec: matching "Elizabeth A. Rodriguez" at "123 Main St" to "Liz Rodriguez" at "123 Main Street Apt 2B".

In [ ]:
# Load the trained model as a DuplicateDetector
detector = DuplicateDetector.from_checkpoint("siamese_model.pt", DEVICE)
print("DuplicateDetector loaded successfully.")

In [ ]:
# Demo 1: Score the exact scenario from the project description
# "Elizabeth A. Rodriguez" at "123 Main St" vs "Liz Rodriguez" at "123 Main Street Apt 2B"

record_a = {
    "first_name": "Elizabeth",
    "middle_name": "A",
    "last_name": "Rodriguez",
    "name_suffix": "",
    "street_address": "123 Main St",
    "city": "Charlotte",
    "zip5": "28205",
    "birth_year": "1985",
}

record_b = {
    "first_name": "Liz",
    "middle_name": "",
    "last_name": "Rodriguez",
    "name_suffix": "",
    "street_address": "123 Main Street Apt 2B",
    "city": "Charlotte",
    "zip5": "28205",
    "birth_year": "1985",
}

prob = detector.score_single_pair(record_a, record_b)
print(f"Match probability: {prob:.4f}")
print(f"  Record A: {record_a['first_name']} {record_a['middle_name']} {record_a['last_name']} @ {record_a['street_address']}")
print(f"  Record B: {record_b['first_name']} {record_b['last_name']} @ {record_b['street_address']}")
print(f"  → {'MATCH ✓' if prob >= 0.5 else 'NO MATCH ✗'} (confidence: {prob:.1%})")

In [ ]:
# Demo 2: Test a battery of hard cases the model should handle
test_cases = [
    # (description, record1, record2, expected)
    (
        "Nickname + address abbreviation (should match)",
        {"first_name": "William", "middle_name": "J", "last_name": "Thompson", "name_suffix": "", "street_address": "456 Oak Avenue", "city": "Raleigh", "zip5": "27601", "birth_year": "1972"},
        {"first_name": "Bill", "middle_name": "", "last_name": "Thompson", "name_suffix": "", "street_address": "456 Oak Ave", "city": "Raleigh", "zip5": "27601", "birth_year": "1972"},
        "MATCH",
    ),
    (
        "Typo in first name (should match)",
        {"first_name": "Robert", "middle_name": "", "last_name": "Davis", "name_suffix": "", "street_address": "789 Pine St", "city": "Durham", "zip5": "27701", "birth_year": "1990"},
        {"first_name": "Roebrt", "middle_name": "", "last_name": "Davis", "name_suffix": "", "street_address": "789 Pine St", "city": "Durham", "zip5": "27701", "birth_year": "1990"},
        "MATCH",
    ),
    (
        "Jr. vs Sr. same name (should NOT match)",
        {"first_name": "John", "middle_name": "A", "last_name": "Smith", "name_suffix": "JR", "street_address": "100 Elm St", "city": "Charlotte", "zip5": "28202", "birth_year": "1995"},
        {"first_name": "John", "middle_name": "A", "last_name": "Smith", "name_suffix": "SR", "street_address": "100 Elm St", "city": "Charlotte", "zip5": "28202", "birth_year": "1965"},
        "NO MATCH",
    ),
    (
        "Married name with hyphen (should match)",
        {"first_name": "Elizabeth", "middle_name": "", "last_name": "Miller", "name_suffix": "", "street_address": "200 Main St", "city": "Greensboro", "zip5": "27401", "birth_year": "1988"},
        {"first_name": "Liz", "middle_name": "", "last_name": "Miller-Davis", "name_suffix": "", "street_address": "200 Main St", "city": "Greensboro", "zip5": "27401", "birth_year": "1988"},
        "MATCH",
    ),
    (
        "Same name, completely different address (should match - person moved)",
        {"first_name": "Maria", "middle_name": "L", "last_name": "Garcia", "name_suffix": "", "street_address": "500 Cedar Blvd", "city": "Charlotte", "zip5": "28205", "birth_year": "1976"},
        {"first_name": "Maria", "middle_name": "L", "last_name": "Garcia", "name_suffix": "", "street_address": "3200 Willow Creek Dr", "city": "Raleigh", "zip5": "27612", "birth_year": "1976"},
        "MATCH",
    ),
    (
        "Different people, same address (roommates - should NOT match)",
        {"first_name": "James", "middle_name": "", "last_name": "Wilson", "name_suffix": "", "street_address": "800 Park Ave Apt 3A", "city": "Durham", "zip5": "27701", "birth_year": "1982"},
        {"first_name": "Sarah", "middle_name": "", "last_name": "Chen", "name_suffix": "", "street_address": "800 Park Ave Apt 3A", "city": "Durham", "zip5": "27701", "birth_year": "1985"},
        "NO MATCH",
    ),
]

print("=== Hard Case Battery Test ===\n")
for desc, r1, r2, expected in test_cases:
    prob = detector.score_single_pair(r1, r2)
    predicted = "MATCH" if prob >= 0.5 else "NO MATCH"
    status = "✓" if predicted == expected else "✗"
    print(f"  {status} {desc}")
    print(f"    {r1['first_name']} {r1['last_name']} @ {r1['street_address']}")
    print(f"    {r2['first_name']} {r2['last_name']} @ {r2['street_address']}")
    print(f"    Prob: {prob:.4f} → {predicted} (expected: {expected})\n")

In [ ]:
# Demo 3: Batch duplicate detection — simulate new registrations arriving
# Take a sample of real records, perturb them, and see if the model finds the originals

from src.augmentation import augment_record

rng = np.random.default_rng(2026)

# Pick 10 random records as "new registrations" (perturbed versions of real people)
demo_seeds = df.sample(n=10, random_state=2026)
demo_new_records = []

for _, row in demo_seeds.iterrows():
    base = {f: str(row.get(f, "")) for f in ["first_name", "middle_name", "last_name", "name_suffix", "street_address", "city", "zip5", "birth_year"]}
    perturbed, applied = augment_record(base, rng, n_perturbations=2)
    perturbed["id"] = f"new_{row['id']}"
    perturbed["original_id"] = row["id"]
    demo_new_records.append(perturbed)

new_df = pd.DataFrame(demo_new_records)
print("=== Simulated New Registrations (perturbed) ===")
for _, r in new_df.iterrows():
    orig_row = df[df["id"] == r["original_id"]].iloc[0]
    print(f"  Original: {orig_row['first_name']} {orig_row['last_name']} @ {orig_row['street_address']}")
    print(f"  New reg:  {r['first_name']} {r['last_name']} @ {r['street_address']}")
    print()

# Run duplicate detection
results = detector.find_duplicates(
    query_records=new_df,
    database_records=df,
    top_k=5,
    threshold=0.3,
    normalize=True,
)

print(f"\n=== Duplicate Detection Results ===")
print(f"Found {len(results)} candidate matches above threshold 0.3")
if len(results) > 0:
    display(results[["query_id", "match_id", "match_probability", "query_name", "match_name", "query_address", "match_address"]].head(20))

## Summary

This pipeline implements:

1. **Preprocessing**: Name/address/ZIP normalization with suffix, directional, and unit canonicalization
2. **Silver/Gold labels**: Proxy ground truth from exact-match groups + birth-year validation
3. **Pair generation**: Positives, hard negatives, block negatives, and DPL/NDPL integration with entity-level splitting
4. **Synthetic augmentation**: Rule-based perturbations with train/test-held-out split for anti-overfitting evaluation
5. **Siamese char-CNN**: Character-level embeddings → CNN encoder → pairwise comparison → match probability
6. **Two-stage evaluation**: Candidate retrieval (Recall@K) + pair classification (F1, AUC, FPR/FNR)

### Key design decisions:
- **Rule-based augmentation** instead of LLM-based: faster, reproducible, controllable
- **Character-level embeddings**: naturally handles typos, abbreviations, nicknames — better than pretrained LMs for this task
- **Anti-overfitting**: held-out perturbation types prove generalization
- **Entity-level splitting**: prevents leakage between train/test